# Countdown problem for GEMMA

## Imports

In [ ]:
from huggingface_hub import login

login()

import os
import re
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset, concatenate_datasets
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl.experimental.gold import GOLDConfig, GOLDTrainer

## GPU and devices

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

torch.cuda.empty_cache()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Number of visible GPUs: {torch.cuda.device_count()}")

## Model loading with 4-bit quantization and LORA

In [ ]:
model_name = "google/gemma-3-1b-it"
print("Loading model in 4-bit...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float32,
)

model = prepare_model_for_kbit_training(model)

print("Adding LoRA adapter...")
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)


tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Model loaded")
print(f"   Trainable params: {model.num_parameters(only_trainable=True):,}")
print(f"   Model device: {model.device}")


## Baseline test (before training)

In [ ]:
print("\n" + "="*50)
print("BASELINE: Model BEFORE training")
print("="*50)

prompt = "Using the numbers [75, 80, 90, 24], create an equation that equals 61. Use each number exactly once."
messages = [{"role": "user", "content": prompt}]

formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False)
inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))
print("="*50 + "\n")

## Dataset loading and preparation

In [ ]:
print("Loading verified datasets...")
ds1 = load_dataset("HuggingFaceTB/Countdown-Task-GOLD", "verified_Qwen2.5-7B-Instruct", split="train")
ds2 = load_dataset("HuggingFaceTB/Countdown-Task-GOLD", "verified_Qwen3-4B-Instruct-2507", split="train")

ds = concatenate_datasets([ds1, ds2])
print(f"Total size: {len(ds)} examples")

ds_split = ds.train_test_split(test_size=0.1, seed=42)

print(f"Train size: {len(ds_split['train'])}")
print(f"Test size: {len(ds_split['test'])}")

## Convert roles for GEMMA

In [ ]:
def convert_roles_for_gemma(example):
    messages = example["messages"].copy()

    user_content_parts = []
    model_content = None

    for msg in messages:
        if msg["role"] == "system":
            user_content_parts.append(f"[System]: {msg['content']}")
        elif msg["role"] == "user":
            user_content_parts.append(msg["content"])
        elif msg["role"] == "assistant":
            model_content = msg["content"]

    converted_messages = []

    if user_content_parts:
        combined_user = "\n\n".join(user_content_parts)
        converted_messages.append({
            "role": "user",
            "content": combined_user
        })

    if model_content:
        converted_messages.append({
            "role": "model",
            "content": model_content
        })

    return {"messages": converted_messages}


ds_split["train"] = ds_split["train"].map(convert_roles_for_gemma)
ds_split["test"] = ds_split["test"].map(convert_roles_for_gemma)

print("="*50)
print("Train sample after conversion:")
print(ds_split["train"][0]["messages"])
print(f"\nTarget: {ds_split['train'][0]['target']}")
print(f"Nums: {ds_split['train'][0]['nums']}")
print("="*50)

## Formatting function for SFT

In [ ]:
def formatting_func(example):
    """Применяет chat_template Gemma к сообщениям"""
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

## SFT Training

In [ ]:
torch.cuda.empty_cache()

print("\nMoving model to cuda:0...")
model = model.to("cuda:0")
print(f"   Model now on: {model.device}")

sft_config = SFTConfig(
    output_dir="./sft_output",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_steps=10,
    eval_strategy="steps",
    eval_steps=10,
    dataloader_pin_memory=False,
    fp16=False,
    bf16=False,
    report_to="none",
    optim="paged_adamw_8bit",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)
TRAIN_SIZE = 1000
EVAL_SIZE = 50

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=ds_split["train"].select(range(TRAIN_SIZE)),
    eval_dataset=ds_split["test"].select(range(EVAL_SIZE)),
    processing_class=tokenizer,
    formatting_func=formatting_func,
)

print("\nStarting SFT training...")
print(f"   Train examples: {len(trainer.train_dataset):,}")
print(f"   Eval examples: {len(trainer.eval_dataset):,}")
print(f"   Epochs: {sft_config.num_train_epochs}")
print(f"   Save every: {sft_config.save_steps} steps")
print(f"   Eval every: {sft_config.eval_steps} steps")
print("="*50)

trainer.train()
print("\nSFT training completed!")

model.save_pretrained("./gemma-countdown-lora-sft")
tokenizer.save_pretrained("./gemma-countdown-lora-sft")
print("Model saved to ./gemma-countdown-lora-sft")

## GOLD training

In [ ]:
from trl.experimental.gold import GOLDConfig, GOLDTrainer

os.environ["TRL_EXPERIMENTAL_SILENCE"] = "1"
torch.cuda.empty_cache()

print("Loading teacher model Qwen2.5-3B...")

teacher_name = "Qwen/Qwen2.5-3B-Instruct"

teacher_model = AutoModelForCausalLM.from_pretrained(
    teacher_name,
    torch_dtype=torch.float16,
    device_map="auto",
)

teacher_tokenizer = AutoTokenizer.from_pretrained(teacher_name)
gold_config = GOLDConfig(
    output_dir="./gold_output",
    
    use_uld_loss=True,
    teacher_tokenizer_name_or_path=teacher_name,
    uld_use_hybrid_loss=True,
    
    uld_hybrid_matched_weight=0.8,
    uld_hybrid_unmatched_weight=0.2,
    
    lmbda=0.5,
    beta=0.5,
    
    max_steps=500,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    warmup_steps=10,
    lr_scheduler_type="cosine",
    
    max_completion_length=256,
    
    logging_steps=5,
    save_steps=25,
    eval_strategy="steps",
    eval_steps=25,
    fp16=False,
    bf16=False,
    report_to="none",
    optim="paged_adamw_8bit",
    disable_dropout=True,
)

print("\nStarting GOLD training...")
print(f"   Teacher: {teacher_name} (FP16, 3B)")
print(f"   Student: google/gemma-3-1b-it (4-bit + LoRA)")
print(f"   Steps: {gold_config.max_steps}")
print(f"   learning_rate: {gold_config.learning_rate}")
print(f"   lmbda: {gold_config.lmbda}, beta: {gold_config.beta}")
print(f"   uld_hybrid_matched_weight: {gold_config.uld_hybrid_matched_weight}")
print("="*50)

trainer_gold = GOLDTrainer(
    model=model,
    teacher_model=teacher_model,
    args=gold_config,
    train_dataset=ds_split["train"].select(range(5000)),
    eval_dataset=ds_split["test"].select(range(500)),
    processing_class=tokenizer,
)

try:
    trainer_gold.train()
    print("\nGOLD training completed!")
    
    print("\nFinal metrics:")
    if hasattr(trainer_gold, 'state') and trainer_gold.state.log_history:
        for log in trainer_gold.state.log_history:
            if 'loss' in log and 'eval_loss' not in log:
                print(f"   Training loss: {log.get('loss', '?'):.4f}")
            if 'eval_loss' in log:
                print(f"   Validation loss: {log.get('eval_loss', '?'):.4f}")
    
except Exception as e:
    print(f"\nError: {e}")
    import traceback
    traceback.print_exc()

### Has problems with resources

## Generate submission

In [ ]:
def extract_equation(text):
    answer_match = re.search(r'<answer>(.*?)</answer>', text, re.DOTALL)
    if answer_match:
        equation = answer_match.group(1).strip()
        equation = re.sub(r'\s+', ' ', equation)
        return equation

    lines = text.split('\n')
    for line in reversed(lines):
        if '=' in line and any(op in line for op in ['+', '-', '*', '/']):
            eq_part = line.split('=')[0].strip()
            if eq_part:
                return eq_part

    return ""

print("\nLoading test data...")
test_dataset = load_dataset("HuggingFaceTB/Countdown-Task-GOLD", "test", split="test")
print(f"Test examples: {len(test_dataset)}")

print("\nGenerating predictions...")
predictions = []

for i, example in enumerate(tqdm(test_dataset, desc="Examples")):
    try:
        nums = example["nums"]
        target = example["target"]
        
        prompt = f"Using the numbers {nums}, create an equation that equals {target}. Use each number exactly once."
        messages = [{"role": "user", "content": prompt}]
        
        formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
        
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.1,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
        
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        equation = extract_equation(response)
        equation = equation.replace('×', '*').replace('÷', '/')
        
        predictions.append({"id": i, "equation": equation if equation else "0"})
    except Exception as e:
        print(f"Error on example {i}: {e}")
        predictions.append({"id": i, "equation": "0"})

df = pd.DataFrame(predictions)
df.to_csv("submission.csv", index=False)

print("\n" + "="*50)
print("Submission file created!")
print(f"   Total predictions: {len(df)}")
print(f"   Format: {df.columns.tolist()}")
print("\nFirst 5 rows:")
print(df.head())
print("="*50)